# Sorting the conferences and HSF trainings via "date" field

In [1]:
import json
import pandas as pd

## Sorting with pandas

In [ ]:
confs = pd.read_json("_data/Conference.json")
confs.dropna(inplace=True)
confs

In [ ]:
confs.sort_values("date", ascending=False)

In [ ]:
result = confs.sort_values("date", ascending=False).to_json(
    orient="records", date_format="iso"
)
parsed = json.loads(result)
parsed

In [ ]:
# writing to file called sorted_confs.json
json.dump(parsed, open("_data/sorted_confs.json", "w"), indent=4)

Write to a script


In [ ]:
%%writefile conferences.py
import pandas as pd
import json

confs = pd.read_json("_data/Conference.json")
confs.dropna(inplace=True)
result = confs.sort_values("date", ascending=False).to_json(
    orient="records", date_format="iso"
)
parsed = json.loads(result)
parsed
json.dump(parsed, open("_data/sorted_confs.json", "w"), indent=4)

In [ ]:
! python conferences.py

## Now for sorting the Trainings

In [ ]:
mentored = pd.read_json("_data/mentored.json")
mentored.drop_duplicates(keep=False, inplace=True)
mentored.dropna(inplace=True)
mentored.sort_values("date", ascending=False, inplace=True)

In [ ]:
parsed = json.loads(mentored.to_json(orient="records", date_format="iso"))
parsed
# json.dump(parsed, open("_data/sorted_mentored.json", "w"), indent=4)

# result = mentored.sort_values("date", ascending=False).to_json(
#     orient="records", date_format="iso"
# )
# # parsed = json.loads(result)
# # parsed

In [ ]:
%%writefile mentored.py
import pandas as pd
import json

mentored = pd.read_json("_data/mentored.json")
mentored.drop_duplicates(keep=False, inplace=True)
mentored.dropna(inplace=True)
mentored.sort_values("date", ascending=False, inplace=True)
parsed = json.loads(mentored.to_json(orient="records", date_format="iso"))

json.dump(parsed, open("_data/sorted_mentored.json", "w"), indent=4)

In [ ]:
! python mentored.py

## Creating the bibfile from inspire

In [ ]:
import requests, time

pages = [1, 2, 3]
all_bibtex = ""

for page in pages:
    url = f"https://inspirehep.net/api/literature?sort=mostrecent&size=50&page={page}&q=a Guillermo.Fidalgo.1&format=bibtex"
    # print(url)
    print(f"Getting {page = } ")
    inspire = requests.get(
        url,
    ).text
    time.sleep(1)
    # print(inspire)
    all_bibtex += inspire + "\n"
print(all_bibtex)
# url = "https://inspirehep.net/api/literature?sort=mostrecent&size=1000&page=1&search_type=hep-author-publication&author=2054153_Guillermo Fidalgo"
# url = "https://inspirehep.net/api/literature/?sort=mostrecent&q=Guillermo%20Fidalgo&size=250&page=1&format=bibtex"

In [ ]:
# response = requests.get('https://inspirehep.net/api/literature?sort=mostrecent&size=250&page=1&q=Guillermo%20Fidalgo')
# url = response.json()['links']['bibtex']
# print(url)
# # print(response['links'])
# inspire = requests.get(url)

In [ ]:
# test script from GPT
import requests
import time

# Base search endpoint
url = "https://inspirehep.net/api/literature"
query = "a Guillermo.Fidalgo.1"

params = {
    "sort": "mostrecent",
    "size": 50,  # Lower size ensures the backend won't give a 502
    "page": 1,
    "q": query,
}

all_bibtex = ""

while True:
    print(f"Querying page {params['page']} for JSON index...")
    response = requests.get(url, params=params)

    if response.status_code != 200:
        print(f"Error {response.status_code} while fetching records.")
        break

    data = response.json()
    hits = data.get("hits", {}).get("hits", [])

    # Stop looping if no more search results return
    if not hits:
        print("Completed processing all pages.")
        break

    print(f"Extracting BibTeX for {len(hits)} papers on this page...")
    for hit in hits:
        # Extract the specialized BibTeX link generated by INSPIRE
        links = hit.get("links", {})
        bibtex_url = links.get("bibtex")

        if bibtex_url:
            try:
                # This fetches the raw pre-compiled string for the single record
                bib_response = requests.get(bibtex_url)
                if bib_response.status_code == 200:
                    all_bibtex += bib_response.text + "\n"
                else:
                    print(f" Failed to get BibTeX for record {hit.get('id')}")
            except Exception as e:
                print(f" Request failed for record {hit.get('id')}: {e}")

            # Respect the API speed limit (15 requests per 5 seconds)
            time.sleep(1)

    # Advance to the next page of the search index
    params["page"] += 1

# Write everything out to a .bib file
if all_bibtex:
    with open("fidalgo_references.bib", "w", encoding="utf-8") as f:
        f.write(all_bibtex)
    print("\n Success! 'fidalgo_references.bib' has been created.")
else:
    print("\n No BibTeX entries were successfully retrieved.")

Making it into a script

In [ ]:
%%writefile createbib.py 
import requests, time
pages = [1,2,3]
all_bibtex = ""

for page in pages:
    url = f"https://inspirehep.net/api/literature?sort=mostrecent&size=50&page={page}&q=a Guillermo.Fidalgo.1&format=bibtex"
    # print(url)
    print(f"Getting {page = } ")
    inspire = requests.get(url,).text
    time.sleep(1)
    # print(inspire)
    all_bibtex += inspire + "\n"

zenodo = r"""
@misc{alexander_moreno_briceno_2022_7115834,
  author       = {Alexander Moreno Briceño and Aman Goel and Guillermo Antonio Fidalgo Rodriguez},
  title        = {Teaching Python the Sustainable Way: Lessons Learned at HSF Training},
  month        = sep,
  year         = 2022,
  publisher    = {Zenodo},
  doi          = {10.5281/zenodo.7115834},
  url          = {https://doi.org/10.5281/zenodo.7115834}
}
"""

# Write everything out to a .bib file
if all_bibtex:
    with open("latex_files/bibfile.bib","w",encoding="utf-8") as f:
        f.write(all_bibtex)
        f.write(zenodo)
    print("\n Success! 'latex_files/bibfile.bib' has been created.")
else:
    print("\n No BibTeX entries were successfully retrieved.")




In [ ]:
! python createbib.py

## Making tests with Pandoc

Just run
```shell
pandoc -C _data/Publications.bib -t markdown_strict \
--csl american-physics-society.csl \
-o _includes/myPubs.md
```

In [ ]:
! pandoc -C ./latex_files/bibfile.bib -t markdown_strict \
--csl ./latex_files/american-physics-society.csl \
 -o _includes/deleteME.md

In [ ]:
%%writefile Fixbib.py

def fixbib(data):
    for id,line in enumerate(data):
        if "$" in line : 
            line = line.replace("$","$$").replace(r"\_","_")
            print(f'Fixed line {id} with "{line}" ')
            data[id] = line
        if "{{s}_" in line:
            line = line.replace("{{s}_","{s_")
            print(f'Fixed line {id} with "{line}" ')
            data[id] = line
        # if "" in line:
        #     line = line.replace("</span>","")
        #     print(f'Fixed line {id} with "{line}" ')
        #     data[id] = line


with open("_includes/allPubs.md") as f:
    data = f.readlines()

fixbib(data)
with open("_includes/allPubs.md","w") as f:
    f.writelines(data)

# Making a pdf instead of markdown

In [ ]:
! pandoc -C -s latex_files/bibfile.bib \
    --csl latex_files/american-physics-society.csl \
    -o latex_files/texfile.tex

In [ ]:
%%writefile Fixtex.py

text = r"""
\usepackage{mathrsfs}
\author{Guillermo Fidalgo}
\date{\today}
\title{Publication List}
\begin{document}
\maketitle
"""

characters = {
    "η": r"\eta",
    "ψ": r"\psi",
    "ν": r"\nu",
    "γ": r"\gamma",
    "τ": r"\tau",
    "μ": r"\mu",
    "\u2006": r" ~ ",
    "Υ": r"\Upsilon",
    "χ": r'\chi'
}
# nitems = len(characters)


def fixbib(data):
    for id, line in enumerate(data):
        if r"\author{}" in line:
            data[id : id + 5] = text
            print("added author, title, date and maketitle in lines", id, id + 5)

        for k, v in characters.items():
            if k in line:
                line1 = line.replace(k, v)
                print(f'Fixed line {id} from \n"{line:^30}"\n to \n"{line1:^30}" ')
                data[id] = line1
                line = line1


with open("latex_files/texfile.tex") as f:
    data = f.readlines()

fixbib(data)
with open("latex_files/texfile.tex", "w") as f:
    f.writelines(data)


In [ ]:
characters = {
    "ψ": r"\psi",
    "ν": r"\nu",
    "γ": r"\gamma",
    "τ": r"\tau",
    "μ": r"\mu",
    "\u2006": r" ~ ",
}

In [ ]:
characters

In [ ]:
for k, v in characters.items():
    if k in "the τ is not found":
        print(f"replacing line with {k}")

# Now on to fixing the rest

In [ ]:
! python Fixtex.py

# Quick local run

In [4]:
! python createbib.py
! pandoc -d latex_files/pandoc-defaults.yaml
! python Fixtex.py

Getting page = 1 
Getting page = 2 
Getting page = 3 

 Success! 'latex_files/bibfile.bib' has been created.
added author, title, date and maketitle in lines 85 90
Fixed line 305 from 
"boson decay H \(\to\) Z\(γ\) in proton-proton collisions at \(\sqrt{s}\)
"
 to 
"boson decay H \(\to\) Z\(\gamma\) in proton-proton collisions at \(\sqrt{s}\)
" 
Fixed line 312 from 
"asymmetry of coherent J/\(ψ\) photoproduction in hadronic PbPb
"
 to 
"asymmetry of coherent J/\(\psi\) photoproduction in hadronic PbPb
" 
Fixed line 406 from 
"fragmentation properties of jets containing \(Υ\)(nS) mesons in
"
 to 
"fragmentation properties of jets containing \(\ensuremath{\Upsilon}\)(nS) mesons in
" 
Fixed line 420 from 
"\(γγ\to ττ\) cross section and constraints on the anomalous magnetic
"
 to 
"\(\gamma\gamma\to ττ\) cross section and constraints on the anomalous magnetic
" 
Fixed line 420 from 
"\(\gamma\gamma\to ττ\) cross section and constraints on the anomalous magnetic
"
 to 
"\(\gamma\gamma\to \